# Bronze Work/Bronze Layer Incremental
- Incremental Bronze ingestion with rerun-safe watermark logic

# Step 1 -- Imports and Setup

This cell imports the PySpark and Delta helpers used in the notebook, switches to the correct catalog,and makes sure the **Bronze Schema** exists before we start loading the data

In [0]:
from pyspark.sql import functions as f
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
spark.sql("use catalog ecomdatabricks")

In [0]:
spark.sql("create schema if not exists bronze_schema")

# Step 2 -- Bronze Control table
This table stores the **watermark** for each source table.
It helps the pipeline remember:
- the latest timestamp already processed
- the latest primary key processed at the timestamp.
- how many rows were written in the latest run.

this is what makes the Bronze load incremental and rerun-safe.

In [0]:
spark.sql(
  """
    create table if not exists ecomdatabricks.bronze_schema.ingestion_control(
      layer string,
      table_name string,
      ts_col string,
      pk_col string,
      last_successful_ts timestamp,
      last_successful_pk bigint,
      last_run_id string,
      rows_written bigint,
      run_status string,
      updated_at timestamp
    )
    using delta
  """
)

# Step 3 -- Source table configuration

This cell defines which source tables will be loaded into Bronze and which columns should be used as:
- **Primary Key**
- **timestamp / watermark column**

it also creates a unique `bronze_run_id` for the current pipeline run.

In [0]:
tables_config = {
  "orders"    : {"pk_col": "order_id", "ts_col": "updated_at"},
  "products"  : {"pk_col": "product_id", "ts_col": "updated_at"},
  "payments"  : {"pk_col": "payment_id", "ts_col": "processed_at"}
}

bronze_run_id = str(uuid.uuid4())
print("current bronze run id: ",bronze_run_id )

# Step 4 -- Helper Functions

This cell contains reusable functions:

- `get_last_successful_watermark()` reads the last processed watermark from the control table
- `upsert_bronze_control()`  updates the control table after a sucessful Bronze load



These functions keep the main load logic cleaner and easier to understand.

In [0]:
def get_last_successful_watermark(table_name:str):
  ctrl = (
    spark.table("ecomdatabricks.bronze_schema.ingestion_control")
    .filter(
      (f.col("layer")=="bronze") &
      (f.col("table_name")==table_name) &
      (f.col("run_status") =="success")
    )
    .orderBy(f.col("updated_at").desc())
    .limit(1)
  )

  rows = ctrl.collect()

  if not rows:
    return None
  
  return rows[0]["last_successful_ts"], rows[0]["last_successful_pk"]

In [0]:
def upsert_bronze_control(table_name, ts_col, pk_col, last_ts, last_pk, rows_written, run_id):
    control_df = spark.createDataFrame(
      [
        (
          "bronze",
          table_name,
          ts_col,
          pk_col,
          last_ts,
          int(last_pk) if last_pk is not None else None,
          run_id,
          int(rows_written),
          "success",
          datetime.utcnow()
        )
      ],
      schema = """
        layer string,
        table_name string,
        ts_col string,
        pk_col string,
        last_successful_ts timestamp,
        last_successful_pk bigint,
        last_run_id string,
        rows_written bigint,
        run_status string,
        updated_at timestamp
      """
    )
    dt = DeltaTable.forName(spark, "ecomdatabricks.bronze_schema.ingestion_control")
    (dt.alias("t")
        .merge(control_df.alias("s"),
        "t.table_name = s.table_name and t.layer = s.layer")
        .whenMatchedUpdate(set={
            "ts_col": "s.ts_col",
            "pk_col": "s.pk_col",
            "last_successful_ts": "s.last_successful_ts",
            "last_successful_pk": "s.last_successful_pk",
            "last_run_id": "s.last_run_id",
            "rows_written": "s.rows_written",
            "run_status": "s.run_status",
            "updated_at": "s.updated_at"
        })
        .whenNotMatchedInsertAll()
        .execute()
        
    )

# Step 5 -- Bronze Incremental load loop
This is the main bronze logic
For each table the notebook will:

1. reads the last watermark.
2. reads the source SQL table.
3. filters only **new / changed rows**.
4. adds Bronze audit column.
5. appends the rows into the Bronze Delta table.
6. updates the control table.


this is the core incremental loading logic.


In [0]:
for table_name, cfg in tables_config.items():
  pk_col = cfg["pk_col"]
  ts_col = cfg["ts_col"]

  source_table = f"ecommerce_sql_connection_catalog.dbo.{table_name}"
  target_table = f"ecomdatabricks.bronze_schema.{table_name}_raw"
  watermark = get_last_successful_watermark(table_name)
  if watermark is None:
    last_successful_ts, last_successful_pk = None, None
  else:
    last_successful_ts, last_successful_pk = watermark
  print(f"\n*** Processing {table_name} ***")
  print(f"Last Successful ts: {last_successful_ts}")
  print(f"Last Successful pk:  {last_successful_pk}")

  source_df = spark.read.table(source_table).withColumn(ts_col, f.col(ts_col).cast("timestamp"))

  if last_successful_ts is None:
    rows_to_load = source_df
  else:
    rows_to_load = source_df.filter(
      (f.col(ts_col) > f.lit(last_successful_ts)) |
      (
        (f.col(ts_col) == f.lit(last_successful_ts)) &
        (f.col(pk_col).cast("long") > f.lit(int(last_successful_pk)))
      )
    )

  rows_to_load = rows_to_load.withColumn("bronze_ingested_at", f.current_timestamp())\
                  .withColumn("bronze_run_id", f.lit(bronze_run_id))\
                  .withColumn("bronze_source_table", f.lit(source_table))

  rows_count = rows_to_load.count()
  print(f"{table_name} rows to load: {rows_count}")

  if rows_count == 0:
    print(f"No rows for {table_name}.")
    continue

  rows_to_load.write.format("delta").mode("append").saveAsTable(target_table)

  max_ts = rows_to_load.agg(f.max(ts_col).alias("max_ts")).collect()[0]["max_ts"]
  max_pk = rows_to_load.filter(f.col(ts_col) == f.lit(max_ts)).agg(f.max(pk_col).cast("long").alias("max_pk")).collect()[0]["max_pk"]

  print(f"Max ts: {max_ts}")
  print(f"Max pk: {max_pk}")

  upsert_bronze_control(
    table_name,
    ts_col,
    pk_col,
    max_ts,
    max_pk,
    rows_count,
    bronze_run_id
  )
  print(f"wrote {rows_count} rows to {target_table}")

# Step 6 -- Quick Validation

This is the final cell prints the bronze row counts and displayes the control table so you can verify the incremental logic working correctly.

In [0]:
print("Orders Bronze Count : ", spark.sql("select count(*) from ecomdatabricks.bronze_schema.orders_raw").collect()[0][0])
print("Products Bronze Count : ", spark.sql("select count(*) from ecomdatabricks.bronze_schema.products_raw").collect()[0][0])
print("Payments Bronze Count : ", spark.sql("select count(*) from ecomdatabricks.bronze_schema.payments_raw").collect()[0][0])
display(spark.sql("select * from ecomdatabricks.bronze_schema.ingestion_control").orderBy("table_name"))